# Test Google Drive filename access

## Goal

Verify that the `RESource.google_drive` module can access the Google Drive account **eliasinul@gmail.com** through an rclone remote, mount it read-only, and resolve a file by its name. OAuth tokens stay in rclone's local configuration and are never stored in this notebook.

## Setup

Run these commands in a terminal once before enabling the live test:

```bash
rclone config
# Choose: n (new remote) -> name: gdrive -> storage: drive
# Complete browser OAuth and select eliasinul@gmail.com
rclone config userinfo gdrive: --json
```

Use browser-based OAuth when prompted. Do not paste OAuth tokens, the rclone configuration, or file contents into this notebook. The local mount directory must be empty.

### 1. Import the module

In [1]:
from pathlib import Path
import json
import shutil
import subprocess

from RESource.google_drive import (
    AmbiguousDriveFileError,
    GoogleDriveMount,
    GoogleDriveMountError,
)

### 2. Set test parameters

Set `TEST_FILENAME` to an existing file's exact basename. If Drive contains duplicate names, set `TEST_RELATIVE_PATH` to the path below the remote root. Change `RUN_LIVE_TEST` to `True` only after configuring rclone.

In [2]:
EXPECTED_ACCOUNT = "eliasinul@gmail.com"
RCLONE_REMOTE = "gdrive:"
TEST_FILENAME = ""  # Example: "weather_2024.csv"
TEST_RELATIVE_PATH = ""  # Example: "RESource/input/weather_2024.csv"
RUN_LIVE_TEST = False

working_directory = Path.cwd().resolve()
repository_root = (
    working_directory.parent if working_directory.name == "notebooks" else working_directory
)
MOUNT_POINT = repository_root / "data" / "tmp" / "google-drive-test"

print(f"Expected Google account: {EXPECTED_ACCOUNT}")
print(f"Remote: {RCLONE_REMOTE}")
print(f"Temporary mount point: {MOUNT_POINT}")
print(f"Live test enabled: {RUN_LIVE_TEST}")

Expected Google account: eliasinul@gmail.com
Remote: gdrive:
Temporary mount point: /home/eliasinul/work/RESource/data/tmp/google-drive-test
Live test enabled: False


## Steps

### 3. Check rclone, remote type, OAuth account, and FUSE

This check does not list Drive files. It verifies that the requested remote exists, asks the backend for the authenticated user, confirms the expected email when the backend supplies it, and checks that Linux FUSE mounting is available.

In [3]:
rclone_binary = shutil.which("rclone")
if rclone_binary is None:
    raise RuntimeError("rclone is not installed or is not on PATH")

configured = subprocess.run(
    [rclone_binary, "listremotes"],
    check=True,
    capture_output=True,
    text=True,
).stdout.splitlines()
remote_configured = RCLONE_REMOTE in configured

fuse_available = Path("/dev/fuse").exists() if shutil.which("fusermount3") else True
if not RUN_LIVE_TEST:
    print(f"Configured {RCLONE_REMOTE} remote: {remote_configured}")
    print(f"FUSE available: {fuse_available}")
    print("Live OAuth check is disabled.")
else:
    if not remote_configured:
        raise RuntimeError(
            f"{RCLONE_REMOTE!r} is not configured. Run 'rclone config', create a remote "
            "named 'gdrive' with backend 'drive', and authorize eliasinul@gmail.com."
        )
    if not fuse_available:
        raise RuntimeError(
            "FUSE is unavailable: /dev/fuse does not exist. Run this notebook in a Linux/WSL "
            "environment with FUSE enabled before testing rclone mount."
        )
    user_check = subprocess.run(
        [rclone_binary, "config", "userinfo", RCLONE_REMOTE, "--json"],
        capture_output=True,
        text=True,
        timeout=60,
    )
    if user_check.returncode != 0:
        message = user_check.stderr.strip().splitlines()[-1] if user_check.stderr.strip() else "unknown error"
        raise RuntimeError(
            f"Google Drive OAuth check failed: {message}. Re-run 'rclone config reconnect gdrive:'."
        )
    user_info = json.loads(user_check.stdout)
    authenticated_email = next(
        (str(value) for key, value in user_info.items() if key.casefold() in {"email", "emailaddress"}),
        None,
    )
    if authenticated_email and authenticated_email.casefold() != EXPECTED_ACCOUNT.casefold():
        raise RuntimeError(
            f"Wrong Google account: expected {EXPECTED_ACCOUNT}, got {authenticated_email}"
        )
    print(f"OAuth access succeeded for {authenticated_email or EXPECTED_ACCOUNT}.")

Configured gdrive: remote: False
FUSE available: True
Live OAuth check is disabled.


### 4. Mount read-only and resolve a filename

The mount exists only inside the `with` block and is automatically released afterward. The cell reports metadata, not file contents.

In [4]:
requested_file = TEST_RELATIVE_PATH or TEST_FILENAME
if not RUN_LIVE_TEST:
    print("Live mount test skipped. Set RUN_LIVE_TEST=True when ready.")
elif not requested_file:
    raise ValueError("Set TEST_FILENAME or TEST_RELATIVE_PATH before running the live test")
else:
    try:
        with GoogleDriveMount(RCLONE_REMOTE, MOUNT_POINT, read_only=True) as drive:
            resolved_path = drive.path(requested_file)
            relative_path = resolved_path.relative_to(MOUNT_POINT)
            print(f"Resolved file: {relative_path}")
            print(f"Size: {resolved_path.stat().st_size:,} bytes")
            print(f"Readable: {resolved_path.is_file()}")
    except AmbiguousDriveFileError as error:
        print(f"Duplicate filename detected: {error}")
        print("Set TEST_RELATIVE_PATH to select the intended file.")
    except GoogleDriveMountError as error:
        raise RuntimeError(f"Mount test failed: {error}") from error

Live mount test skipped. Set RUN_LIVE_TEST=True when ready.


## Checks

In [5]:
if RUN_LIVE_TEST:
    assert requested_file, "A filename or relative path is required"
    assert not GoogleDriveMount(RCLONE_REMOTE, MOUNT_POINT).is_mounted, (
        "Mount still active after the test; stop the rclone process before continuing"
    )
    print("PASS: OAuth access, filename resolution, and automatic unmount completed.")
else:
    print("SAFE MODE: notebook structure passed; live Google Drive checks were not run.")

SAFE MODE: notebook structure passed; live Google Drive checks were not run.


## Next Steps

After a successful test, keep `RUN_LIVE_TEST = False` before committing the notebook so outputs do not reveal private Drive names or paths. Use the same remote and filename-resolution pattern in RESource workflows. Keep production mounts read-only unless a reviewed workflow explicitly requires writes.